In [1]:
import os, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm

warnings.filterwarnings("ignore")


In [8]:
class CFG:
    BASE_DIR   = Path("/kaggle/input/competitions/birdclef-2026")
    MODEL_DIR  = Path("/kaggle/input/datasets/mariia222/birdclef2026-models")
    OUTPUT_DIR = Path("/kaggle/working")
    TAXON_CSV  = BASE_DIR / "taxonomy.csv"
    SAMPLE_SUB = BASE_DIR / "sample_submission.csv"
    TEST_DIR   = BASE_DIR / "test_soundscapes"

    MODEL      = "convnext_small"
    TAG        = "approach5_convnext_small_full"

    SR         = 32_000
    DURATION   = 5
    N_FFT      = 1024
    HOP_LENGTH = 512
    N_MELS     = 128
    FMIN       = 50
    FMAX       = 14_000

    BATCH_SIZE = 64
    NUM_WORKERS = 2
    TTA        = 3     
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"



In [9]:
_full_path = CFG.MODEL_DIR / f"{CFG.TAG}.pth"
_fold_path = CFG.MODEL_DIR / f"{CFG.TAG}.pth"

if _full_path.exists():
    WEIGHTS_FILE = _full_path
elif _fold_path.exists():
    WEIGHTS_FILE = _fold_path
else:
    raise FileNotFoundError(
        f"Не знайдено ні {_full_path.name}, ні {_fold_path.name} "
        f"у {CFG.MODEL_DIR}.\n"
        f"Наявні файли: {list(CFG.MODEL_DIR.glob('*.pth'))}"
    )

print(f"Device     : {CFG.DEVICE}")
print(f"TTA passes : {CFG.TTA}")

taxonomy   = pd.read_csv(CFG.TAXON_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

all_species = sorted(taxonomy["primary_label"].tolist())
NUM_CLASSES = len(all_species)
print(f"Classes    : {NUM_CLASSES}")
print(f"Sub rows   : {len(sample_sub):,}")

Device     : cpu
TTA passes : 3
Classes    : 234
Sub rows   : 3


In [10]:
def parse_row_id(row_id):
    parts    = row_id.rsplit("_", 1)
    end_time = int(parts[1])
    fname    = parts[0] + ".ogg"
    return fname, end_time

meta = pd.DataFrame({"row_id": sample_sub["row_id"]})
meta[["filename", "end_time"]] = meta["row_id"].apply(
    lambda r: pd.Series(parse_row_id(r))
)
meta["start_time"] = meta["end_time"] - CFG.DURATION
meta["filepath"]   = meta["filename"].apply(lambda f: str(CFG.TEST_DIR / f))

print(f"Test files : {meta['filename'].nunique()}")
print(f"Segments   : {len(meta):,}")

def load_clip(filepath, start):
    n = CFG.SR * CFG.DURATION
    
    if not os.path.exists(filepath):
        return np.zeros(n, dtype=np.float32)
        
    offset = max(0.0, float(start))
    try:
        y, _ = librosa.load(filepath, sr=CFG.SR, offset=offset, duration=CFG.DURATION)
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return np.zeros(n, dtype=np.float32)
        
    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y[:n].astype(np.float32)

Test files : 1
Segments   : 3


In [11]:
def compute_melspec(y):
    mel    = librosa.feature.melspectrogram(
        y=y, sr=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS, fmin=CFG.FMIN, fmax=CFG.FMAX,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0).astype(np.float32)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db

In [12]:
class TestDataset(Dataset):
    def __init__(self, df, add_noise=False):
        self.df        = df.reset_index(drop=True)
        self.add_noise = add_noise 

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y   = load_clip(row["filepath"], row["start_time"])

        if self.add_noise:
            y = y + np.random.normal(0, 0.003, y.shape).astype(np.float32)

        mel = compute_melspec(y)
        mel = torch.from_numpy(np.stack([mel, mel, mel]))
        return mel, row["row_id"]


class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.MODEL, pretrained=False,
            in_chans=3, num_classes=0, global_pool="avg"
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, NUM_CLASSES),
        )
    def forward(self, x): return self.head(self.backbone(x))


In [13]:
def load_model(weights_path):
    model = BirdModel().to(CFG.DEVICE)
    state = torch.load(weights_path, map_location=CFG.DEVICE)
    model.load_state_dict(state)
    model.eval()
    print(f"  Loaded: {weights_path.name}")
    return model


In [14]:
@torch.no_grad()
def predict(model):
    all_preds, all_ids = [], []

    for tta_pass in range(CFG.TTA):
        noise = (tta_pass > 0) 
        ds = TestDataset(meta, add_noise=noise)
        dl = DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

        pass_preds, pass_ids = [], []
        for mels, row_ids in dl:
            mels = mels.to(CFG.DEVICE)
            with autocast():
                out = torch.sigmoid(model(mels)).cpu().numpy()
            pass_preds.append(out)
            if tta_pass == 0:
                pass_ids.extend(row_ids)

        all_preds.append(np.concatenate(pass_preds))
        if tta_pass == 0:
            all_ids = pass_ids

        print(f"  TTA pass {tta_pass+1}/{CFG.TTA} done")

    avg = np.mean(all_preds, axis=0)
    return all_ids, avg

model = load_model(WEIGHTS_FILE)

row_ids, preds = predict(model)
del model; gc.collect(); torch.cuda.empty_cache()

print(f"\nPreds shape : {preds.shape}")
print(f"Stats — min={preds.min():.4f}  max={preds.max():.4f}  mean={preds.mean():.4f}")

submission = pd.DataFrame(preds, columns=all_species)
submission.insert(0, "row_id", row_ids)

submission = sample_sub[["row_id"]].merge(submission, on="row_id", how="left")
submission[all_species] = submission[all_species].fillna(0.0)

assert submission.shape == sample_sub.shape, (
    f"Shape mismatch! Got {submission.shape}, expected {sample_sub.shape}"
)

out = CFG.OUTPUT_DIR / "submission.csv"
submission.to_csv(out, index=False)
print(f"  Shape: {submission.shape}")
print(submission.iloc[:3, :5].to_string())

  Loaded: approach5_convnext_small_full.pth
  TTA pass 1/3 done
  TTA pass 2/3 done
  TTA pass 3/3 done

Preds shape : (3, 234)
Stats — min=0.0000  max=0.0288  mean=0.0050
  Shape: (3, 235)
                                    row_id   1161364    116570   1176823   1491113
0   BC2026_Test_0001_S05_20250227_010002_5  0.000291  0.000909  0.000301  0.003000
1  BC2026_Test_0001_S05_20250227_010002_10  0.000302  0.000957  0.000298  0.003188
2  BC2026_Test_0001_S05_20250227_010002_15  0.000232  0.000675  0.000311  0.002078
